In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 8 - WEEK 12 BAYESIAN OPTIMISATION
# Run from inside week12/
# ============================================================
#
# Week 11 produced a NEW incumbent and the GP prediction was
# extremely well calibrated locally (~ +0.08 sigma residual).
#
# Therefore Week 12 should remain exploitation-focused:
# - centre on the Week 11 incumbent
# - refit the GP using all observations
# - use a controlled ARD-informed local trust region
# - prioritise posterior mean and low-beta UCB
# - use EI / beta=1 mainly as exploration diagnostics
# - inspect one-coordinate and recent-direction behaviour
#
# Candidates are rounded to six decimals BEFORE the
# near-duplicate test so diagnostics match portal submissions.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 12 data
# ------------------------------------------------------------

X = np.load(
    "function8/initial_inputs.npy"
)

Y = np.load(
    "function8/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 11 calibration check
# ------------------------------------------------------------

week11_pred_mean = (
    9.99858691
)

week11_pred_std = (
    0.0136990
)

week11_actual = (
    9.9996456622444
)

week11_error = (
    week11_actual
    - week11_pred_mean
)

week11_z_error = (
    week11_error
    / week11_pred_std
)

print("\n================================")
print("WEEK 11 CALIBRATION CHECK")
print("================================")

print(
    "Predicted mean:",
    week11_pred_mean
)

print(
    "Predicted std :",
    week11_pred_std
)

print(
    "Actual        :",
    week11_actual
)

print(
    "\nPrediction error:",
    week11_error
)

print(
    "\nError / predicted std:",
    week11_z_error
)


# ------------------------------------------------------------
# 3. Recent strong observations
# ------------------------------------------------------------

week9_x = np.array([
    0.117062,
    0.183135,
    0.104304,
    0.120414,
    0.770644,
    0.497652,
    0.179588,
    0.558852
])

week9_y = (
    9.9940246476926
)


week10_x = np.array([
    0.146187,
    0.187173,
    0.143955,
    0.098243,
    0.778452,
    0.497627,
    0.170757,
    0.594843
])

week10_y = (
    9.9891379221651
)


week11_x = np.array([
    0.107323,
    0.149041,
    0.129094,
    0.157441,
    0.815313,
    0.493175,
    0.201585,
    0.586041
])

week11_y = (
    9.9996456622444
)


print("\n================================")
print("RECENT LOCAL POINTS")
print("================================")

print("\nWeek 9:")
print(
    week9_x,
    "->",
    week9_y
)

print("\nWeek 10:")
print(
    week10_x,
    "->",
    week10_y
)

print("\nWeek 11 / current best:")
print(
    week11_x,
    "->",
    week11_y
)


# ------------------------------------------------------------
# 4. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(8) * 0.4,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Week 11 succeeded and calibration was excellent.
# Allow a meaningful but still conservative local neighbourhood.

empirical_cap = min(
    1.15 * nearest_distance,
    0.065
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed trust region
# ------------------------------------------------------------
#
# Smaller GP lengthscale -> tighter search width.
# Dimensions at the upper lengthscale bound naturally receive
# a wider local allowance, subject to the empirical cap.
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.04 * lengthscales,
    0.020,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 12 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 8. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        650000,
        8
    )
)

# Portal precision first
candidates = np.round(
    candidates,
    6
)

tree = cKDTree(
    X
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 10. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)

print(
    "predicted improvement =",
    mu[mean_idx] - best_y
)


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"beta={beta}",
        "\n candidate =",
        candidates[idx],
        "\n mean =",
        round(
            mu[idx],
            8
        ),
        "\n std =",
        round(
            sigma[idx],
            8
        ),
        "\n UCB =",
        round(
            UCB[idx],
            8
        ),
        "\n"
    )


# ------------------------------------------------------------
# 13. Distance from incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 14. Trust-region boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 15. Axis-profile diagnostics
# ------------------------------------------------------------
#
# Hold seven dimensions at the Week 11 incumbent and change
# one coordinate at a time.
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

for j in range(8):

    values = np.linspace(
        lower[j],
        upper[j],
        35000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile = np.round(
        profile,
        6
    )

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.01
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 16. Week 9 -> Week 11 direction diagnostic
# ------------------------------------------------------------
#
# Week 9 and Week 11 are both exceptionally strong points.
#
# Search along their empirical direction to determine whether
# the updated GP prefers:
# - continuing beyond Week 11
# - staying close to Week 11
# - moving slightly back toward Week 9
# ------------------------------------------------------------

direction = (
    week11_x
    - week9_x
)

direction = (
    direction
    / np.linalg.norm(direction)
)

pos_limits = []
neg_limits = []

for j in range(8):

    if direction[j] > 0:

        pos_limits.append(
            (upper[j] - best_x[j])
            / direction[j]
        )

        neg_limits.append(
            (best_x[j] - lower[j])
            / direction[j]
        )

    elif direction[j] < 0:

        pos_limits.append(
            (lower[j] - best_x[j])
            / direction[j]
        )

        neg_limits.append(
            (best_x[j] - upper[j])
            / direction[j]
        )


t_pos_max = min(
    pos_limits
)

t_neg_max = min(
    neg_limits
)

t_values = np.linspace(
    -t_neg_max,
    t_pos_max,
    60000
)

line_candidates = (
    best_x
    + t_values[:, None]
    * direction
)

line_candidates = np.round(
    line_candidates,
    6
)

line_distance, _ = tree.query(
    line_candidates,
    k=1
)

line_valid = line_candidates[
    line_distance > 0.01
]

if len(line_valid) > 0:

    line_mu, line_sigma = gp.predict(
        line_valid,
        return_std=True
    )

    line_idx = np.argmax(
        line_mu
    )

    print("\n================================")
    print("WEEK 9 -> WEEK 11 DIRECTION")
    print("================================")

    print(
        "Best line candidate =",
        line_valid[line_idx]
    )

    print(
        "mean =",
        line_mu[line_idx]
    )

    print(
        "std =",
        line_sigma[line_idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            line_valid[line_idx]
            - best_x
        )
    )

else:

    print(
        "\nNo admissible Week 9 -> Week 11 line points."
    )

DATA
X shape: (51, 8)
Y shape: (51,)

Current best:
[0.107323 0.149041 0.129094 0.157441 0.815313 0.493175 0.201585 0.586041] -> 9.9996456622444

Y range:
min = 5.5921933895401965
max = 9.9996456622444
std = 1.2150063149911206

WEEK 11 CALIBRATION CHECK
Predicted mean: 9.99858691
Predicted std : 0.013699
Actual        : 9.9996456622444

Prediction error: 0.0010587522444005515

Error / predicted std: 0.07728682709690865

RECENT LOCAL POINTS

Week 9:
[0.117062 0.183135 0.104304 0.120414 0.770644 0.497652 0.179588 0.558852] -> 9.9940246476926

Week 10:
[0.146187 0.187173 0.143955 0.098243 0.778452 0.497627 0.170757 0.594843] -> 9.9891379221651

Week 11 / current best:
[0.107323 0.149041 0.129094 0.157441 0.815313 0.493175 0.201585 0.586041] -> 9.9996456622444


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co


GP FIT

Fitted kernel:
0.857**2 * Matern(length_scale=[1.28, 2, 1.01, 2, 2, 2, 1.35, 2], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[1.27861651 2.         1.0062348  2.         2.         2.
 1.35403995 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.15596895 0.09971223 0.1981888  0.09971223 0.09971223 0.09971223
 0.14728108 0.09971223]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.0805063016539699

Empirical cap: 0.065

WEEK 12 TRUST REGION
Centre:
[0.107323 0.149041 0.129094 0.157441 0.815313 0.493175 0.201585 0.586041]

Half-widths:
[0.05114466 0.065      0.04024939 0.065      0.065      0.065
 0.0541616  0.065     ]

Lower:
[0.05617834 0.084041   0.08884461 0.092441   0.750313   0.428175
 0.1474234  0.521041  ]

Upper:
[0.15846766 0.214041   0.16934339 0.222441   0.880313   0.558175
 0.2557466  0.651041  ]

Candidates after portal rounding + duplicate filtering:
650000

PRIMARY EI
candidate = [0.057745 0.091517 0.096381 0.096121 0.805633 0.443382 0.

In [2]:
# ============================================================
# FINAL FUNCTION 8 - WEEK 12 SELECTION
# ============================================================
#
# Week 11 produced the new incumbent and had exceptionally
# good local GP calibration (~ +0.08 sigma residual).
#
# For Week 12:
# - the highest posterior-mean candidate is interior
# - its uncertainty is low
# - beta=0.05 supports the same general local region
# - axis profiles independently support several of its main
#   directions (lower x1, higher x2, lower x5, higher x7)
#
# EI and higher-beta UCB move substantially farther from the
# incumbent and are increasingly uncertainty-driven.
#
# Therefore select the local highest posterior-mean candidate.
# ============================================================

week12_candidate = np.array([
    0.088567,
    0.167731,
    0.131357,
    0.163034,
    0.800372,
    0.500019,
    0.206649,
    0.588480
])

print("================================")
print("FUNCTION 8 - WEEK 12 FINAL")
print("================================")

print("\nCandidate:")
print(week12_candidate)

# GP prediction
final_mu, final_sigma = gp.predict(
    week12_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])

print("\nPredicted improvement over observed best:")
print(final_mu[0] - best_y)

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week12_candidate - best_x
    )
)

# Check six-decimal submitted point against all previous data
nearest_dist, nearest_idx = tree.query(
    week12_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(nearest_dist[0] > 0.01)

portal = "-".join(
    f"{x:.6f}"
    for x in week12_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 8 - WEEK 12 FINAL

Candidate:
[0.088567 0.167731 0.131357 0.163034 0.800372 0.500019 0.206649 0.58848 ]

Predicted mean:
10.000546149977326

Predicted std:
0.004227693735792463

Predicted improvement over observed best:
0.0009004877329257255

Distance from current best:
0.032236207717409926

Nearest observed-point distance:
0.032236207717409926

Nearest observed point:
[0.107323 0.149041 0.129094 0.157441 0.815313 0.493175 0.201585 0.586041]

Duplicate rule satisfied:
True

Portal format:
0.088567-0.167731-0.131357-0.163034-0.800372-0.500019-0.206649-0.588480
